In [1]:
import numpy as np
import pandas as pd

from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OneHotEncoder

print("Libraries imported successfully!")

Libraries imported successfully!


In [2]:
df = pd.read_csv("../data/application_train.csv")

print("Dataset loaded successfully!")
print("Shape:", df.shape)

Dataset loaded successfully!
Shape: (307511, 122)


In [3]:
X = df.drop(columns=["TARGET"])
y = df["TARGET"]

print("X shape:", X.shape)
print("y shape:", y.shape)

X shape: (307511, 121)
y shape: (307511,)


In [4]:
X_train, X_temp, y_train, y_temp = train_test_split(
    X,
    y,
    test_size=0.40,
    stratify=y,
    random_state=42
)

X_val, X_test, y_val, y_test = train_test_split(
    X_temp,
    y_temp,
    test_size=0.50,
    stratify=y_temp,
    random_state=42
)

print("X_train:", X_train.shape)
print("X_val:", X_val.shape)
print("X_test:", X_test.shape)

print("y_train:", y_train.shape)
print("y_val:", y_val.shape)
print("y_test:", y_test.shape)

X_train: (184506, 121)
X_val: (61502, 121)
X_test: (61503, 121)
y_train: (184506,)
y_val: (61502,)
y_test: (61503,)


In [5]:
numerical_features = X_train.select_dtypes(
    include=["int64", "float64"]
).columns.tolist()

categorical_features = X_train.select_dtypes(
    include=["object", "category"]
).columns.tolist()

print("Numerical features:", len(numerical_features))
print("Categorical features:", len(categorical_features))
print("Total features:", len(numerical_features) + len(categorical_features))

Numerical features: 105
Categorical features: 16
Total features: 121


/var/folders/cy/8z0w251d7nl6jf5zvlt_78300000gn/T/ipykernel_76762/3021389004.py:5: Pandas4Warning: For backward compatibility, 'str' dtypes are included by select_dtypes when 'object' dtype is specified. This behavior is deprecated and will be removed in a future version. Explicitly pass 'str' to `include` to select them, or to `exclude` to remove them and silence this warning.
See https://pandas.pydata.org/docs/user_guide/migration-3-strings.html#string-migration-select-dtypes for details on how to write code that works with pandas 2 and 3.
  categorical_features = X_train.select_dtypes(


In [6]:
train_missing = X_train.isnull().sum()

train_missing = (
    train_missing[train_missing > 0]
    .sort_values(ascending=False)
)

print("Features with missing values:", len(train_missing))
print(train_missing.head(20))

Features with missing values: 67
COMMONAREA_MEDI             128839
COMMONAREA_AVG              128839
COMMONAREA_MODE             128839
NONLIVINGAPARTMENTS_MEDI    127994
NONLIVINGAPARTMENTS_MODE    127994
NONLIVINGAPARTMENTS_AVG     127994
FONDKAPREMONT_MODE          126059
LIVINGAPARTMENTS_MODE       126018
LIVINGAPARTMENTS_MEDI       126018
LIVINGAPARTMENTS_AVG        126018
FLOORSMIN_MODE              125175
FLOORSMIN_MEDI              125175
FLOORSMIN_AVG               125175
YEARS_BUILD_MODE            122602
YEARS_BUILD_MEDI            122602
YEARS_BUILD_AVG             122602
OWN_CAR_AGE                 121853
LANDAREA_AVG                109328
LANDAREA_MEDI               109328
LANDAREA_MODE               109328
dtype: int64


In [7]:
for data in [X_train, X_val, X_test]:
    data["DAYS_EMPLOYED"] = data["DAYS_EMPLOYED"].replace(
        365243, np.nan
    )

print("DAYS_EMPLOYED anomaly handled.")

DAYS_EMPLOYED anomaly handled.


In [8]:
for data in [X_train, X_val, X_test]:
    
    data["AGE_YEARS"] = -data["DAYS_BIRTH"] / 365.25
    
    data["EMPLOYMENT_YEARS"] = (
        -data["DAYS_EMPLOYED"] / 365.25
    )
    
    data["CREDIT_INCOME_RATIO"] = (
        data["AMT_CREDIT"] /
        data["AMT_INCOME_TOTAL"]
    )
    
    data["ANNUITY_INCOME_RATIO"] = (
        data["AMT_ANNUITY"] /
        data["AMT_INCOME_TOTAL"]
    )
    
    data["CREDIT_GOODS_RATIO"] = (
        data["AMT_CREDIT"] /
        data["AMT_GOODS_PRICE"]
    )

print("Application-level features created.")

Application-level features created.


/var/folders/cy/8z0w251d7nl6jf5zvlt_78300000gn/T/ipykernel_76762/4281718132.py:3: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  data["AGE_YEARS"] = -data["DAYS_BIRTH"] / 365.25
/var/folders/cy/8z0w251d7nl6jf5zvlt_78300000gn/T/ipykernel_76762/4281718132.py:5: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  data["EMPLOYMENT_YEARS"] = (
/var/folders/cy/8z0w251d7nl6jf5zvlt_78300000gn/T/ipykernel_76762/4281718132.py:9: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, 

In [9]:
engineered_features = [
    "AGE_YEARS",
    "EMPLOYMENT_YEARS",
    "CREDIT_INCOME_RATIO",
    "ANNUITY_INCOME_RATIO",
    "CREDIT_GOODS_RATIO"
]

X_train[engineered_features].describe().T

,count,mean,std,min,25%,50%,75%,max
AGE_YEARS,184506.0,43.887027,11.953844,21.007529,33.943874,43.082820,53.891170,68.993840
EMPLOYMENT_YEARS,151281.0,6.528409,6.403443,-0.000000,2.094456,4.511978,8.695414,48.139630
CREDIT_INCOME_RATIO,184506.0,3.956901,2.690627,0.004808,2.017858,3.261024,5.153846,49.227200
ANNUITY_INCOME_RATIO,184499.0,0.180881,0.094739,0.000224,0.114550,0.162700,0.228457,1.451571
CREDIT_GOODS_RATIO,184345.0,1.122879,0.124353,0.150000,1.000000,1.118800,1.198000,6.000000


In [10]:
def add_application_features(data):
    data = data.copy()

    data = data.assign(
        AGE_YEARS=-data["DAYS_BIRTH"] / 365.25,
        EMPLOYMENT_YEARS=-data["DAYS_EMPLOYED"] / 365.25,
        CREDIT_INCOME_RATIO=(
            data["AMT_CREDIT"] / data["AMT_INCOME_TOTAL"]
        ),
        ANNUITY_INCOME_RATIO=(
            data["AMT_ANNUITY"] / data["AMT_INCOME_TOTAL"]
        ),
        CREDIT_GOODS_RATIO=(
            data["AMT_CREDIT"] / data["AMT_GOODS_PRICE"]
        )
    )

    return data


X_train = add_application_features(X_train)
X_val = add_application_features(X_val)
X_test = add_application_features(X_test)

print("Application-level features added successfully!")
print("Train shape:", X_train.shape)
print("Validation shape:", X_val.shape)
print("Test shape:", X_test.shape)

Application-level features added successfully!
Train shape: (184506, 126)
Validation shape: (61502, 126)
Test shape: (61503, 126)


In [11]:
print("Train missing values:", X_train.isnull().sum().sum())
print("Validation missing values:", X_val.isnull().sum().sum())
print("Test missing values:", X_test.isnull().sum().sum())

Train missing values: 5552248
Validation missing values: 1852738
Test missing values: 1858517


In [12]:
numerical_features = X_train.select_dtypes(
    include=["int64", "float64"]
).columns.tolist()

categorical_features = X_train.select_dtypes(
    include=["object", "category", "str"]
).columns.tolist()

print("Numerical features:", len(numerical_features))
print("Categorical features:", len(categorical_features))
print("Total features:", len(numerical_features) + len(categorical_features))

Numerical features: 110
Categorical features: 16
Total features: 126


In [13]:
numeric_pipeline = Pipeline([
    ("imputer", SimpleImputer(strategy="median"))
])

categorical_pipeline = Pipeline([
    ("imputer", SimpleImputer(strategy="most_frequent")),
    ("encoder", OneHotEncoder(
        handle_unknown="ignore",
        sparse_output=False
    ))
])

preprocessor = ColumnTransformer([
    ("num", numeric_pipeline, numerical_features),
    ("cat", categorical_pipeline, categorical_features)
])

print("Preprocessing pipeline created successfully!")

Preprocessing pipeline created successfully!


In [14]:
X_train_processed = preprocessor.fit_transform(X_train)

X_val_processed = preprocessor.transform(X_val)

X_test_processed = preprocessor.transform(X_test)

print("Preprocessing completed!")
print("Train processed:", X_train_processed.shape)
print("Validation processed:", X_val_processed.shape)
print("Test processed:", X_test_processed.shape)

Preprocessing completed!
Train processed: (184506, 250)
Validation processed: (61502, 250)
Test processed: (61503, 250)


In [15]:
print(
    "Training missing values:",
    np.isnan(X_train_processed).sum()
)

print(
    "Validation missing values:",
    np.isnan(X_val_processed).sum()
)

print(
    "Test missing values:",
    np.isnan(X_test_processed).sum()
)

Training missing values: 0
Validation missing values: 0
Test missing values: 0


In [16]:
negative_samples = (y_train == 0).sum()
positive_samples = (y_train == 1).sum()

scale_pos_weight = negative_samples / positive_samples

print("Negative samples:", negative_samples)
print("Positive samples:", positive_samples)
print("Scale Pos Weight:", scale_pos_weight)

Negative samples: 169611
Positive samples: 14895
Scale Pos Weight: 11.38710976837865


In [17]:
import joblib

joblib.dump(preprocessor, "../data/preprocessor.pkl")

joblib.dump(
    X_train_processed,
    "../data/X_train_processed.pkl"
)

joblib.dump(
    X_val_processed,
    "../data/X_val_processed.pkl"
)

joblib.dump(
    X_test_processed,
    "../data/X_test_processed.pkl"
)

joblib.dump(y_train, "../data/y_train.pkl")
joblib.dump(y_val, "../data/y_val.pkl")
joblib.dump(y_test, "../data/y_test.pkl")

print("Preprocessing artifacts saved successfully!")

Preprocessing artifacts saved successfully!


# Conclusion — Preprocessing

The application dataset was split into training, validation, and test sets using stratified sampling to preserve the imbalanced target distribution.

Five application-level features were engineered:

- `AGE_YEARS`
- `EMPLOYMENT_YEARS`
- `CREDIT_INCOME_RATIO`
- `ANNUITY_INCOME_RATIO`
- `CREDIT_GOODS_RATIO`

Missing numerical values are handled using median imputation, while categorical missing values are handled using the most frequent category. Categorical variables are one-hot encoded with unknown categories ignored.

The preprocessing pipeline was fitted exclusively on the training data and subsequently applied to the validation and test sets to prevent data leakage.

The resulting application-level feature matrix contains 250 features after one-hot encoding.

The target remains highly imbalanced, with approximately 11.39 negative samples for every positive sample. This imbalance will be considered during model training using an appropriate class-weighting strategy.

Historical credit and repayment datasets will be incorporated in the feature-engineering stage before final model development.